# Retinex (MSRCP)
Multi-Scale Retinex with Chromaticity Preservation (Petro et al., 2014). Enhancement baseline: removes the low-frequency illumination (where haze lives) and keeps the colour ratios of the input.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE     = "/content/drive/MyDrive"
SRC       = f"{DRIVE}/O-HAZE"          # original O-HAZE: hazy/ + GT/
DATA_ROOT = f"{DRIVE}/O-HAZE-1600"     # resized copy, shared by all methods and the evaluation
OUT       = f"{DRIVE}/outputs/retinex" # dehazed images, same file names as hazy/

MAX_IMAGES = 5             # first N pairs (None = all 45)
RESIZE = 1600              # long side; O-HAZE is ~3000x4000

In [ ]:
#resize hazy + GT once (skipped if DATA_ROOT already exists), same as the other method notebooks
import os, glob
from PIL import Image

def resize_long(img, long_side):
    if long_side <= 0 or max(img.size) <= long_side:
        return img
    s = long_side / max(img.size)
    return img.resize((round(img.size[0] * s), round(img.size[1] * s)), Image.BICUBIC)

if not os.path.isdir(f"{DATA_ROOT}/hazy"):
    os.makedirs(f"{DATA_ROOT}/hazy"); os.makedirs(f"{DATA_ROOT}/GT")
    pairs = list(zip(sorted(glob.glob(f"{SRC}/hazy/*")), sorted(glob.glob(f"{SRC}/GT/*"))))[:MAX_IMAGES]
    for h, g in pairs:
        hazy = resize_long(Image.open(h).convert('RGB'), RESIZE)
        gt   = Image.open(g).convert('RGB').resize(hazy.size, Image.BICUBIC)
        stem = lambda p: os.path.splitext(os.path.basename(p))[0]
        hazy.save(f"{DATA_ROOT}/hazy/{stem(h)}.png"); gt.save(f"{DATA_ROOT}/GT/{stem(g)}.png")

hazy_paths = sorted(glob.glob(f"{DATA_ROOT}/hazy/*"))
os.makedirs(OUT, exist_ok=True)
print(len(hazy_paths), "images:", [os.path.basename(p) for p in hazy_paths])

## Parameters

In [ ]:
SIGMAS = (15, 80, 250)     # Gaussian scales for the illumination estimate (small = detail, large = global)
LOW, HIGH = 1.0, 99.0      # percentiles used to stretch the MSR output

NAME = "retinex"

## Method

In [ ]:
def ssr(img_float, sigma):
    #single-scale retinex: log(image) - log(blurred image = illumination)
    blur = cv2.GaussianBlur(img_float, (0, 0), sigma).reshape(img_float.shape)
    return np.log(img_float) - np.log(blur)


def msr(img_float, sigmas=SIGMAS):
    #average of SSR over several scales
    return sum(ssr(img_float, s) for s in sigmas) / len(sigmas)


def dehaze(bgr):
    #MSR on the intensity only, then scale B, G, R by the same ratio -> colours are preserved
    f = bgr.astype(np.float64) + 1.0
    intensity = f.mean(axis=2)

    msr_int = msr(intensity[..., None])[..., 0]
    lo, hi = np.percentile(msr_int, LOW), np.percentile(msr_int, HIGH)
    int1 = np.clip((msr_int - lo) / (hi - lo + 1e-8), 0, 1) * 254 + 1

    ratio = np.minimum(int1 / intensity, 255.0 / f.max(axis=2))   # no channel above 255
    return np.clip(f * ratio[..., None], 0, 255).astype(np.uint8)

## Preview a single image

In [ ]:
hazy = cv2.imread(hazy_paths[0])
out = dehaze(hazy)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, im, title in zip(ax, [hazy, out], ['hazy', NAME]):
    a.imshow(im[..., ::-1]); a.set_title(title); a.axis('off')
plt.tight_layout(); plt.show()

## Run over the dataset

In [ ]:
import time
for p in hazy_paths:
    hazy = cv2.imread(p)
    t0 = time.time()
    out = dehaze(hazy)
    dt = time.time() - t0
    cv2.imwrite(f"{OUT}/{os.path.basename(p)}", out)
    print(f"{os.path.basename(p)} | {hazy.shape[1]}x{hazy.shape[0]} | {dt:.2f}s")

## Evaluate
Score the outputs with `dehazing/evaluation` (see `dehaze_eval_colab.ipynb`):
```python
per_image, summary = evaluate(DATA_ROOT, {"retinex": OUT})
```